## Tâche 4 — Un premier modèle, et ce qu'il révèle

On jette toutes les colonnes dans une forêt aléatoire, sans réfléchir à ce
qu'elles contiennent, et on regarde ce que le modèle en fait.

Le cas d'usage fixe la règle. Escale Atlantique veut savoir, avant l'arrivée
du client, si sa réservation va être annulée — pour reloger, surréserver,
relancer. Toute colonne remplie après ce moment est interdite au modèle,
même si elle prédit très bien.

La méthode d'enquête est donc celle du brief : pour chaque variable en haut
du classement, se demander à quel moment elle est renseignée dans la vie
d'une réservation.

In [15]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

query = text("""
    SELECT r.*, h.code AS hotel
    FROM reservations r
    JOIN hotels h ON r.hotel_id = h.hotel_id
""")
df = pd.read_sql(query, engine, parse_dates=["arrival_date", "reservation_status_date"])

TEST_MONTHS = 4

# le test commence au 1er jour du 4e mois avant la fin de l'historique
debut_test = (df["arrival_date"].max().to_period("M") - (TEST_MONTHS - 1)).to_timestamp()

train = df[df["arrival_date"] < debut_test].copy()
test = df[df["arrival_date"] >= debut_test].copy()

print(f"train : {len(train)} | test : {len(test)}")

train : 58792 | test : 19911


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# la consigne dit toutes les colonnes : on ne retire que ce qui n'est pas une variable
a_exclure = [
    "is_canceled",              # la cible
    "arrival_date",             # date, exclue par la consigne
    "reservation_status_date",  # date, exclue par la consigne
    "reservation_id",           # un numéro de ligne, aucun sens métier
    "reservation_status",       # fuite de données : équivaut exactement à la cible
]

# encodage sommaire : chaque modalité de chaque colonne texte devient une colonne 0/1
X = pd.get_dummies(train.drop(columns=a_exclure))
y = train["is_canceled"]

print(X.shape)

(58792, 229)


In [17]:
rf = RandomForestClassifier(
    n_estimators=100,  # nombre d'arbres qui votent, plus il y en a plus le résultat est stable
    random_state=42,   # graine du hasard, pour obtenir le même modèle à chaque exécution
    n_jobs=-1,         # utilise tous les cœurs du processeur, ne change que la vitesse
)

# cv=3 : train est coupé en 3, on entraîne sur 2 parts et on mesure sur la 3e, trois fois
# accuracy : part de réponses correctes, annulées et non annulées confondues
scores = cross_val_score(rf, X, y, cv=3, scoring="accuracy")

print(f"accuracy en validation croisée : {scores.round(4)} | moyenne {scores.mean():.4f}")

accuracy en validation croisée : [0.8476 0.8302 0.8151] | moyenne 0.8309


In [18]:
# réentraînement sur tout train
rf.fit(X, y)

# feature_importances_ : part du travail de séparation portée par chaque colonne, somme à 1
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

importances.head(20)

lead_time                        0.092306
country_PRT                      0.076141
deposit_type_Non Refund          0.072982
deposit_type_No Deposit          0.062648
previous_cancellations           0.062106
adr                              0.061885
total_of_special_requests        0.053403
agent                            0.044581
arrival_date_day_of_month        0.041327
arrival_date_week_number         0.040242
stays_in_week_nights             0.031047
stays_in_weekend_nights          0.020782
required_car_parking_spaces      0.018321
assigned_room_type_A             0.017711
booking_changes                  0.016079
market_segment_Groups            0.015261
market_segment_Online TA         0.013576
customer_type_Transient-Party    0.012652
customer_type_Transient          0.012448
adults                           0.012134
dtype: float64

In [19]:
pd.crosstab(train["reservation_status"], train["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,20206
Check-Out,37883,0
No-Show,0,703


### Ce que j'ai trouvé

100 % en validation croisée, ça ne tient pas debout. Personne ne peut savoir
en mars qu'un client annulera en août.

J'ai regardé les importances. Deux colonnes portent 64 % du modèle à elles
seules, et elles sortent de la même variable : reservation_status.

Le croisement avec la cible est sans appel :

| reservation_status | is_canceled = 0 | is_canceled = 1 |
|---|---|---|
| Canceled   |     0 | 20 206 |
| Check-Out  | 37 883 |     0 |
| No-Show    |     0 |    703 |

Pas une exception sur 58 792 lignes. reservation_status n'est pas un indice
d'annulation, c'est l'annulation écrite autrement.

La colonne est remplie à la fin : au départ du client, à son annulation, ou
le jour où il ne se présente pas. Au moment où l'hôtel veut la prédiction,
elle est vide.

Je la retire, avec reservation_status_date qui date ce même statut.

## Tâche 5 — Rejouer la v1 pour avoir une référence

La v1 a déjà ses scores, mais je ne peux pas m'en servir. Elle a été entraînée
sur un découpage aléatoire, donc sur des réservations qui sont dans mon jeu de
test. Je comparerais ma v2 à un modèle qui connaît déjà les réponses.

Je reprends sa recette — build_pipeline() et FEATURES dans
model/train_baseline.py — et je la réentraîne sur mon train à moi. Même recette,
mes données, ma validation croisée.

Le score que j'obtiens là, c'est ma référence.

In [20]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# le notebook tourne depuis notebooks/, la racine du projet est au-dessus
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pour pouvoir importer model.train_baseline
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

query = text("""
    SELECT r.*, h.code AS hotel
    FROM reservations r
    JOIN hotels h ON r.hotel_id = h.hotel_id
""")
df = pd.read_sql(query, engine, parse_dates=["arrival_date", "reservation_status_date"])

TEST_MONTHS = 4

# le test commence au 1er jour du 4e mois avant la fin de l'historique
debut_test = (df["arrival_date"].max().to_period("M") - (TEST_MONTHS - 1)).to_timestamp()

train = df[df["arrival_date"] < debut_test].copy()
test = df[df["arrival_date"] >= debut_test].copy()

print(f"train : {len(train)} | test : {len(test)}")

train : 58792 | test : 19911


Validation croisée

In [21]:
from sklearn.model_selection import cross_validate

from model.train_baseline import FEATURES, TARGET, build_pipeline

# la recette de la v1 prend les colonnes brutes, elle fait l'encodage elle-même
X_train = train[FEATURES]
y_train = train[TARGET]

print(f"{len(FEATURES)} variables : {X_train.shape}")

15 variables : (58792, 15)


In [22]:
scores = cross_validate(
    build_pipeline(), X_train, y_train,
    cv=5,                                          # 5 découpages du jeu d'entraînement
    scoring=["f1", "precision", "recall"],         # plusieurs métriques en un seul passage
    return_train_score=True,                       # pour comparer entraînement et validation
    n_jobs=-1,
)

pd.DataFrame(scores).agg(["mean", "std"]).T.round(4)

,mean,std
fit_time,3.1749,0.0364
score_time,0.1789,0.0025
test_f1,0.7005,0.1245
train_f1,0.8761,0.0117
test_precision,0.7818,0.1651
train_precision,0.9045,0.0144
test_recall,0.6776,0.1777
train_recall,0.8495,0.0099


Étape 2 — Surapprentissage

In [23]:
res = pd.DataFrame(scores)

for m in ["f1", "precision", "recall"]:
    t, v = res[f"train_{m}"].mean(), res[f"test_{m}"].mean()
    print(f"{m:10} train {t:.4f} | validation {v:.4f} | écart {t - v:+.4f}")

f1         train 0.8761 | validation 0.7005 | écart +0.1756
precision  train 0.9045 | validation 0.7818 | écart +0.1226
recall     train 0.8495 | validation 0.6776 | écart +0.1719


Étape 3 — Mini-recherche

In [24]:
from sklearn.model_selection import GridSearchCV

# <nom de l'étape>__<paramètre> : "classifier" est le nom donné au modèle dans le pipeline
grille = {
    "classifier__max_iter": [100, 200],
    "classifier__learning_rate": [0.05, 0.1],
}

recherche = GridSearchCV(build_pipeline(), grille, cv=5, scoring="f1", n_jobs=-1, return_train_score=True)
recherche.fit(X_train, y_train)

cols = ["param_classifier__max_iter", "param_classifier__learning_rate",
        "mean_test_score", "std_test_score", "mean_train_score"]
pd.DataFrame(recherche.cv_results_)[cols].round(4)

,param_classifier__max_iter,param_classifier__learning_rate,mean_test_score,std_test_score,mean_train_score
0,100,0.05,0.7077,0.1027,0.8473
1,200,0.05,0.7065,0.1077,0.8579
2,100,0.10,0.7071,0.1085,0.8590
3,200,0.10,0.7005,0.1114,0.8761


Étape 4 — TimeSeriesSplit

In [25]:
from sklearn.model_selection import TimeSeriesSplit

# TimeSeriesSplit valide toujours sur une période postérieure, les données doivent être triées
train_trie = train.sort_values("arrival_date")

scores_ts = cross_validate(
    build_pipeline(), train_trie[FEATURES], train_trie[TARGET],
    cv=TimeSeriesSplit(n_splits=5),
    scoring=["f1", "precision", "recall"],
    return_train_score=True,
    n_jobs=-1,
)

pd.DataFrame(scores_ts).agg(["mean", "std"]).T.round(4)

,mean,std
fit_time,1.8530,1.1012
score_time,0.1437,0.0155
test_f1,0.7117,0.1465
train_f1,0.9223,0.0301
test_precision,0.8549,0.0497
train_precision,0.9408,0.0152
test_recall,0.6221,0.1787
train_recall,0.9049,0.0447
